# Historical ozone analysis
1973 NYC data, used to demonstrate method only.
Run from the repository root or notebooks directory. Source: canonical R airquality via statsmodels / saved CSV.
Complete cases are primary; missing-data approaches are descriptive sensitivity analyses.
Log-response statistics use log(ppb), so do not rank its AIC/BIC or RMSE against raw-response models.


In [1]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd

from src.api_clients import baseline
from src.modeling import baseline_frame, fit, metrics, missing_comparison, normality, vif

raw = baseline_frame(baseline(True))
clean = raw.dropna()
print(missing_comparison(raw).to_string(index=False))

       Method   n  Ozone SD  Pearson r with Temp
Complete case 111 33.275969             0.698541
         Mean 153 28.693372             0.608742
       Median 153 29.054449             0.600833


In [2]:
from src.modeling import baseline_sensitivity

sensitivity = baseline_sensitivity(raw)
print(sensitivity.to_string(index=False))
print('Imputed responses are descriptive sensitivity analysis only, never live validation targets.')
print('Training R² differs across changed samples; this is not evidence of comparative predictive skill.')

       Method   n       R²  Adjusted R²         AIC         BIC      RMSE  Residual Shapiro p  Imputed ozone targets  Intercept  Temperature coefficient (ppb/°F)  Wind coefficient (ppb/mph)  Solar coefficient (ppb/Langley)
Complete case 111 0.605895     0.594845  996.717103 1007.555224 20.795614        3.617786e-06                      0 -64.342079                          1.652093                   -3.333591                         0.059821
         Mean 153 0.480025     0.469556 1368.273503 1380.395255 20.622851        7.827876e-08                     37 -38.223155                          1.241263                   -2.717250                         0.057747
       Median 153 0.472094     0.461465 1374.416454 1386.538206 21.041038        7.603676e-08                     37 -39.354064                          1.232953                   -2.787086                         0.056958
Imputed responses are descriptive sensitivity analysis only, never live validation targets.
Training R² diff

In [3]:
print(normality(clean).to_string(index=False))
print('Pearson correlation')
print(clean.corr().round(3).to_string())
print('Spearman correlation')
print(clean.corr(method='spearman').round(3).to_string())

Variable        W            p
   ozone 0.873554 2.846396e-08
    temp 0.980067 9.569113e-02
    wind 0.980764 1.098834e-01
   solar 0.932850 2.957179e-05
Pearson correlation
       ozone   temp   wind  solar
ozone  1.000  0.699 -0.612  0.348
temp   0.699  1.000 -0.497  0.294
wind  -0.612 -0.497  1.000 -0.127
solar  0.348  0.294 -0.127  1.000
Spearman correlation
       ozone   temp   wind  solar
ozone  1.000  0.773 -0.605  0.348
temp   0.773  1.000 -0.499  0.210
wind  -0.605 -0.499  1.000 -0.062
solar  0.348  0.210 -0.062  1.000


In [4]:
simple = fit(clean, ['temp'])
multiple = fit(clean)
log_model = fit(clean, log=True)
comparison = pd.DataFrame([metrics(m) for m in [simple, multiple, log_model]], index=['Simple', 'Multiple', 'Log response'])
print(comparison.to_string())
print(pd.DataFrame({'coefficient': multiple.params, 'p': multiple.pvalues}).to_string())
print(vif(clean).to_string(index=False))

                n        R²  Adjusted R²          AIC          BIC       RMSE  Residual Shapiro p
Simple        111  0.487960     0.483262  1021.775145  1027.194205  23.703772        1.522253e-07
Multiple      111  0.605895     0.594845   996.717103  1007.555224  20.795614        3.617786e-06
Log response  111  0.664424     0.655015   168.825058   179.663178   0.499323        5.726116e-02
       coefficient             p
const   -64.342079  6.226638e-03
temp      1.652093  2.423506e-09
wind     -3.333591  1.515934e-06
solar     0.059821  1.123664e-02
Feature      VIF
   temp 1.431367
   wind 1.329070
  solar 1.095253


In [5]:
from src import viz

# Interactive figures display when re-running in Jupyter.
figures = [viz.histogram(clean.ozone, 'Ozone (ppb)'), viz.qq(clean.ozone, 'Ozone (ppb)'), viz.heatmap(clean, 'pearson'), viz.residual(multiple)]
for figure in figures:
    print(figure.layout.title.text)
# Uncomment to view in an interactive notebook:
# for figure in figures: figure.show()

Distribution · ozone
Normal Q–Q · ozone
Pearson correlation
Residual diagnostics
